<p align="center">
• <a href="https://mayzune.com/"><strong>May Zune</strong></a> •
<a href="https://github.com/hellomayzune"><strong>GitHub</strong></a> •
<a href="https://orcid.org/0000-0003-0282-2633"><strong>ORCID</strong></a> •
<a href="https://scholar.google.com/citations?user=LmP8B_4AAAAJ&hl=en"><strong>Google Scholar</strong></a> •
<a href="https://www.researchgate.net/profile/May-Zune"><strong>ResearchGate</strong></a> •
<a href="https://www.linkedin.com/in/mayzune//"><strong>Linkedin</strong></a> •
</p>

# 🤖 Overall Framework | Beginner Mind Diagnostics Tutorial

After week 10, revisited where it started with a beginner's mind. This is the first diagnostics notebook.

The notebook provides a comparative experimental framework for evaluating several different approaches to **surrogate-based black-box optimisation**.

The central distinction is between methods that explicitly model **uncertainty** and those that primarily model the **predicted objective surface**.

* **Gaussian Process Bayesian Optimisation ** explicitly incorporates predictive uncertainty through its acquisition function.
* **Random Forest SMBO** uses disagreement between ensemble trees as an approximate uncertainty measure.
* **Logistic Regression** converts the continuous optimisation problem into a binary classification problem.
* **SVR** directly models the objective surface but does not natively provide uncertainty estimates.
* **MLPRegressor** provides a flexible nonlinear surrogate and directly optimises its predicted surface.

These approaches allow the notebook to investigate how different surrogate models and search strategies behave across black-box functions with varying dimensionality, sample sizes, nonlinearities, and peak structures.


## 📈 Summary of the Evaluated Approaches

| **Method**                | **Surrogate**                    | **Search / Acquisition Strategy** | **Candidate/Search Approach** |
| ------------------------- | -------------------------------- | --------------------------------- | ----------------------------- |
| **Bayesian Optimization** | Gaussian Process + Matérn kernel | Expected Improvement (EI)         | 25 L-BFGS-B random restarts   |
| **Logistic Regression**   | Binary classifier                | Maximum probability of Class 1    | 50,000 random candidates      |
| **Random Forest SMBO**    | Random Forest Regressor          | Upper Confidence Bound (UCB)      | 200,000 random candidates     |
| **SVR**                   | RBF Support Vector Regression    | Direct surrogate maximization     | 50 L-BFGS-B random restarts   |
| **MLP**                   | Two-layer neural network         | Direct surrogate maximization     | 50 L-BFGS-B random restarts   |


### ✅ Overall Ranking

| **Rank** | **Framework**                       | **Overall Assessment** | **Primary Advantage**                                      |
| -------: | ----------------------------------- | ---------------------- | ---------------------------------------------------------- |
|    **1** | **Bayesian Optimization (GP + EI)** | ⭐ **Best**             | Explicit uncertainty + principled exploration/exploitation |
|    **2** | **Random Forest SMBO (UCB)**        | **Very Good**          | Ensemble uncertainty + nonlinear modelling                 |
|    **3** | **MLPRegressor**                    | **Moderate**           | Flexible nonlinear surrogate                               |
|    **4** | **Logistic Regression**             | **Suboptimal**         | Simple and computationally lightweight                     |
|    **5** | **SVR**                             | **Worst**              | Smooth nonlinear surrogate but no uncertainty model        |


## 🏋️ Comparison of Surrogate-Based Optimization Strategies

This Jupyter Notebook compares several **surrogate-based optimization strategies** for finding maximum values of multiple black-box continuous functions (**F1** through **F8**) loaded from an Excel file, `CapstoneFunction_from_Numpy.xlsx`.

The functions have input dimensionalities ranging from **2D to 8D**.

The common objective across all approaches is to:

1. Fit a surrogate model using the existing observations $(X,y)$.
2. Search the bounded domain $[0,1]^d$.
3. Identify a promising input configuration.
4. Return the recommended next input coordinates (`next_x` or `next_query`) for evaluation.

---

## ⚙️ Common Optimization Pipeline

Despite their different surrogate models and search mechanisms, the approaches follow a common structure:

```text
Existing Observations (X, y)
          │
          ▼
    Train Surrogate
          │
          ▼
 Approximate Black-Box
      Function
          │
          ▼
 Search Domain [0, 1]^d
          │
          ▼
 Select Promising Input
          │
          ▼
     next_x / next_query
          │
          ▼
Evaluate Actual Function
```

The principal difference between the methods is **how they represent the black-box function and how they decide which candidate should be evaluated next**.


In [1]:
import pandas as pd

excel_path = 'initial_data/CapstoneFunction_from_Numpy.xlsx'
xls = pd.ExcelFile(excel_path)
print("Sheet names:", xls.sheet_names)

for sheet in xls.sheet_names:
    df = pd.read_excel(excel_path, sheet_name=sheet)
    print(f"\n--- Sheet: {sheet} ---")
    print(df.head())
    print("Shape:", df.shape)

Sheet names: ['F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8']

--- Sheet: F1 ---
    Input_1   Input_2         Output
0  0.319404  0.762959   1.322677e-79
1  0.574329  0.879898   1.033078e-46
2  0.731024  0.733000   7.710875e-16
3  0.840353  0.264732  3.341771e-124
4  0.650114  0.681526  -3.606063e-03
Shape: (10, 3)

--- Sheet: F2 ---
    Input_1   Input_2    Output
0  0.665800  0.123969  0.538996
1  0.877791  0.778628  0.420586
2  0.142699  0.349005 -0.065624
3  0.845275  0.711120  0.293993
4  0.454647  0.290455  0.214965
Shape: (10, 3)

--- Sheet: F3 ---
    Input_1   Input_2   Input_3    Output
0  0.171525  0.343917  0.248737 -0.112122
1  0.242114  0.644074  0.272433 -0.087963
2  0.534906  0.398501  0.173389 -0.111415
3  0.492581  0.611593  0.340176 -0.034835
4  0.134622  0.219917  0.458206 -0.048008
Shape: (15, 4)

--- Sheet: F4 ---
    Input_1   Input_2   Input_3   Input_4     Output
0  0.896981  0.725628  0.175404  0.701694 -22.108288
1  0.889356  0.499588  0.539269  0.508783 -14

# 🎯 Bayesian Optimisation

This script executes **Bayesian Optimization** across multiple black-box functions loaded from an Excel file. It builds a probabilistic surrogate model using a **Gaussian Process (GP)** for each dataset and determines the optimal next input coordinates (`next_x`) to evaluate in order to maximize the function output.

## 1. Expected Improvement (`expected_improvement`)

### Purpose

Calculates the **Acquisition Function** value at candidate points `X`.

### Mechanism

Expected Improvement (EI) measures how much improvement a candidate point is expected to yield over the current maximum observation (`mu_sample_opt`). It balances:

* **Predicted reward** (`μ`)
* **Predictive uncertainty** (`σ`)

This allows the optimization process to balance **exploitation** of promising regions with **exploration** of uncertain regions.

### Exploration Parameter (`xi=0.01`)

The parameter `xi` controls the exploration–exploitation trade-off:

* `xi > 0` encourages exploration of uncertain regions.
* Smaller values place more emphasis on exploiting areas with high predicted values.
* The script uses `xi = 0.01`.

### Division-by-Zero Handling

At points that have already been sampled, the predictive standard deviation can be zero. The script handles this explicitly:

```python
ei[sigma == 0.0] = 0.0
```

This prevents numerical errors when calculating the Expected Improvement.

---

## 2. Acquisition Optimization (`propose_location`)

### Purpose

Finds the input coordinates `x` in the search space `[0, 1]^d` that maximize the **Expected Improvement**.

### L-BFGS-B Optimization

The script uses `scipy.optimize.minimize` with **L-BFGS-B** to optimize the acquisition function.

Because `minimize()` performs minimization, the acquisition function is negated:

```python
-acquisition()
```

Thus, minimizing `-EI(x)` is equivalent to maximizing `EI(x)`.

### Random Restarts (`n_restarts=25`)

To reduce the risk of becoming trapped in a local optimum, the optimizer uses **25 random starting points**.

Each starting point is sampled uniformly throughout the search domain:

```text
x0 ~ Uniform([0, 1]^d)
```

The optimizer runs from each starting location, and the best resulting solution is selected as the recommended next point.

---

## 3. Gaussian Process Regressor (`GaussianProcessRegressor`)

The script uses `GaussianProcessRegressor` as the probabilistic surrogate model for each black-box function.

### Kernel

The Gaussian Process uses a combination of a **Constant Kernel** and a **Matérn kernel**:

```text
C × Matern(ν=2.5)
```

The components serve different purposes:

* **Constant Kernel (`C`)** — models the overall output scale.
* **Matérn Kernel (`ν=2.5`)** — models spatial relationships and provides smooth interpolation between observations.

The Matérn kernel with `ν=2.5` provides a balance between smoothness and flexibility.

### Hyperparameter Bounds

The script uses broadened hyperparameter bounds:

```text
(1e-6, 1e6)
(1e-2, 1e5)
```

These wider limits help prevent optimization issues when fitting datasets with:

* Near-zero output variance
* Flat input dimensions
* Very large or very small characteristic scales

They also reduce the likelihood of `ConvergenceWarning` messages caused by hyperparameters reaching their default bounds.

### Target Normalization (`normalize_y=True`)

The target values are normalized before fitting the Gaussian Process:

```python
normalize_y=True
```

This effectively standardizes the observations to approximately:

```text
μ = 0
σ = 1
```

Normalization improves numerical stability and can make the GP hyperparameter optimization more robust, particularly when different functions have substantially different output scales.

### Numerical Stability (`alpha=1e-4`)

The model adds a small diagonal noise term:

```python
alpha=1e-4
```

This improves numerical stability when solving the Gaussian Process covariance system and helps prevent problems caused by nearly singular covariance matrices.

---

## 4. Data Loop and Output Generation

### Sheet Iteration

The script processes each worksheet in the Excel file independently.

For each sheet:

1. Input variables are extracted into `X`.
2. Function outputs are extracted into `y`.
3. A Gaussian Process model is fitted to the observations.
4. Expected Improvement is optimized.
5. The recommended next input coordinates are stored as `next_x`.
6. The model predicts the expected output at `next_x`.

The resulting information is then collected for all functions.

### Warning Tracker

The Gaussian Process fitting step is wrapped in `warnings.catch_warnings()` so the script can monitor whether hyperparameter optimization converged cleanly.

This allows the script to identify situations where the optimizer encounters warnings such as hyperparameters reaching their specified bounds.

Conceptually:

```python
with warnings.catch_warnings():
    warnings.simplefilter("always")
    gpr.fit(X, y)
```

The warning status can then be tracked alongside the optimization results.

---

## 5. Results Formatting

After processing all Excel sheets, the script generates a Markdown table containing the key optimization results.

The table includes:

| Column                            | Description                                         |
| --------------------------------- | --------------------------------------------------- |
| **Function**                      | Identifier for the black-box function               |
| **Dimensions**                    | Number of input dimensions                          |
| **Samples**                       | Number of observations currently available          |
| **Current Max Output**            | Highest observed function value                     |
| **Next Recommended Query Vector** | Input coordinates selected by Bayesian Optimization |
| **Predicted Expected Output**     | GP-predicted output at the recommended query point  |

The resulting table provides a concise summary of the current optimization state and identifies the **next point that should be evaluated** for each function.

---

## Overall Workflow

The complete optimization process can be summarized as:

```text
Excel File
    │
    ├── Function F1
    ├── Function F2
    ├── Function F3
    └── ...
         │
         ▼
    Extract X and y
         │
         ▼
    Fit Gaussian Process
         │
         ▼
    Calculate Expected Improvement
         │
         ▼
    Optimize Acquisition Function
         │
         ▼
    Select next_x
         │
         ▼
    Predict expected output
         │
         ▼
    Add results to Markdown table
```

#### Summary
The script uses **Gaussian Process regression** to model each black-box function and **Expected Improvement** to determine where the next evaluation should occur. The combination allows the optimizer to intelligently balance **exploration of uncertain regions** with **exploitation of regions that are predicted to produce high function values**.


<div style="background-color: #d4edda; color: #155724; padding: 15px; border-radius: 5px; border: 1px solid #c3e6cb;">
  <strong>Hi ya!!</strong> Start the optimisation.
</div>

In [12]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
import warnings
from sklearn.exceptions import ConvergenceWarning

xls = pd.ExcelFile(excel_path)

def expected_improvement(X, X_sample, Y_sample, gpr, xi=0.01):
    X = np.atleast_2d(X)
    mu, sigma = gpr.predict(X, return_std=True)
    mu_sample_opt = np.max(Y_sample)

    with np.errstate(divide='ignore'):
        imp = mu - mu_sample_opt - xi
        Z = imp / sigma
        ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0

    return ei

def propose_location(acquisition, X_sample, Y_sample, gpr, bounds, n_restarts=25):
    dim = bounds.shape[0]
    min_val = 1e10
    min_x = None

    def min_obj(x):
        return -acquisition(x.reshape(1, -1), X_sample, Y_sample, gpr)

    for x0 in np.random.uniform(bounds[:, 0], bounds[:, 1], size=(n_restarts, dim)):
        res = minimize(min_obj, x0=x0, bounds=bounds, method='L-BFGS-B')
        if res.fun < min_val:
            min_val = res.fun
            min_x = res.x

    return min_x

results = []

for sheet in xls.sheet_names:
    df = pd.read_excel(excel_path, sheet_name=sheet)
    X = df.drop(columns=['Output']).values
    y = df['Output'].values
    dim = X.shape[1]
    
    bounds = np.array([[0.0, 1.0] for _ in range(dim)])
    
    # 1. Expanded length_scale_bounds to (1e-2, 1e5) to handle smooth/flat dimensions
    # 2. Expanded constant_value bounds to (1e-6, 1e6) to handle extreme near-zero variance outputs
    # 3. Enabled normalize_y=True to scale target standard deviations
    kernel = C(1.0, (1e-6, 1e6)) * Matern(length_scale=np.ones(dim), length_scale_bounds=(1e-2, 1e5), nu=2.5)
    gpr = GaussianProcessRegressor(
        kernel=kernel, 
        alpha=1e-4, 
        normalize_y=True, 
        n_restarts_optimizer=10, 
        random_state=42
    )
    
    # Catch any convergence warnings to verify clean execution
    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always", ConvergenceWarning)
        gpr.fit(X, y)
        warning_count = len(caught_warnings)
    
    next_x = propose_location(expected_improvement, X, y, gpr, bounds)
    pred_mean, pred_std = gpr.predict(next_x.reshape(1, -1), return_std=True)
    
    results.append({
        'Function': sheet,
        'Dim': dim,
        'Observations': len(df),
        'Current Max Output': np.max(y),
        'Proposed Next Query': [round(v, 6) for v in next_x],
        'Predicted Output': round(pred_mean[0], 6),
        'Warnings': warning_count
    })

for r in results:
    query_str = ", ".join([f"{v:.6f}" for v in r['Proposed Next Query']])
    c_max = f"{r['Current Max Output']:.6e}" if (abs(r['Current Max Output']) < 1e-3 and r['Current Max Output'] != 0) else f"{r['Current Max Output']:.6f}"
    p_mean = f"{r['Predicted Output']:.6e}" if (abs(r['Predicted Output']) < 1e-3 and r['Predicted Output'] != 0) else f"{r['Predicted Output']:.6f}"
    print(f"| **{r['Function']}** | {r['Dim']}D | {r['Observations']} | {c_max} | `[{query_str}]` | {p_mean} | {r['Warnings']} |")

| **F1** | 2D | 10 | 7.710875e-16 | `[0.794516, 0.707086]` | -3.440000e-04 | 0 |
| **F2** | 2D | 10 | 0.611205 | `[0.706009, 0.047705]` | 0.572148 | 0 |
| **F3** | 3D | 15 | -0.034835 | `[0.976014, 0.000000, 0.000000]` | -0.061121 | 0 |
| **F4** | 4D | 30 | -4.025542 | `[0.162689, 0.565070, 0.771611, 0.498891]` | -12.350004 | 0 |
| **F5** | 4D | 20 | 1088.859618 | `[0.288920, 0.673054, 1.000000, 1.000000]` | 1643.281090 | 0 |
| **F6** | 5D | 20 | -0.714265 | `[0.464583, 0.241972, 0.575161, 1.000000, 0.000000]` | -0.224760 | 0 |
| **F7** | 6D | 30 | 1.364968 | `[0.000000, 0.242022, 0.714053, 0.218015, 0.375317, 0.747545]` | 1.352130 | 0 |
| **F8** | 8D | 40 | 9.598482 | `[0.473921, 0.716397, 0.270943, 0.202254, 0.313827, 0.241501, 0.214913, 0.424851]` | 9.140586 | 1 |


| **Function** | **Dimensions** | **Samples** |     **Current Max Output** | **Next Recommended Query Vector**                                                  | **Predicted Expected Output** |
| ------------ | -------------: | ----------: | -------------------------: | ---------------------------------------------------------------------------------- | ----------------------------: |
| **Fx** | ?D | ?? | see above | see above | see above4 | see above |

<div style="background-color: #d4edda; color: #155724; padding: 15px; border-radius: 5px; border: 1px solid #c3e6cb;">
  <strong>Success!</strong> Bayesian Optimisation Done!
</div>

# 🎯 Surrogate-Based Optimization Using Logistic Regression

This code implements a **surrogate-based optimization strategy** for continuous black-box functions using **Logistic Regression**. Because continuous optimization algorithms typically require a regression model, the code reformulates the maximization problem as a **binary classification problem**.

## Key Workflow

### 1. Data Extraction and Separation

* Iterates through every sheet in an Excel workbook (`xls`), treating each sheet as a separate dataset representing a different function.
* Separates the feature vectors (`X`) from the continuous target values (`df['Output']`).
* Automatically determines the dimensionality (`n_dim`) from the number of input features.

---

### 2. Classification Thresholding

The code calculates the median observed output:

```python
threshold = np.median(y)
```

The continuous target values are then converted into two classes:

* Points with outputs **at or above the median** are assigned class `1` — **high performing**.
* Points with outputs **below the median** are assigned class `0` — **low performing**.

This transforms the original continuous optimization problem into a binary classification problem.

---

### 3. Model Training and Domain Sampling

A `LogisticRegression` classifier is trained to learn the relationship between the input coordinates and whether they correspond to a high-performing region.

The model effectively learns a **linear decision boundary** separating the two classes in the input space.

The code then generates **50,000 uniformly distributed random candidate points** across the bounded hypercube:

```text
[0, 1]^d
```

where `d` is the dimensionality of the function.

---

### 4. Acquisition and Candidate Selection

For every candidate point, the trained model calculates the probability that it belongs to the high-performing class:

```python
model.predict_proba(candidates)[:, 1]
```

The candidate with the highest predicted probability is selected as the recommended next evaluation point:

```text
next_query = candidate with maximum P(Class = 1)
```

In effect, the classifier acts as a simple **surrogate model**, and the predicted probability serves as the acquisition criterion.

---

### 5. Results Compilation

For each function, the script records:

* **Function name**
* **Input dimensionality**
* **Current maximum observed output**
* **Recommended next evaluation point**
* Coordinates formatted to **6 decimal places**

The results from all functions are then compiled into a summary DataFrame and printed without index numbers.

---

### Strengths

* **Lightweight:** Logistic Regression is computationally inexpensive compared with more complex surrogate models.
* **Fast:** Training and prediction are generally very quick, even when evaluating many candidate points.
* **Simple to implement:** The approach requires relatively little code and has few hyperparameters.
* **Useful for global trends:** It can effectively identify broad linear or monotonic directions toward high-value regions.
* **Scales reasonably well:** The model remains relatively inexpensive as the number of candidate points increases.

### Limitations

* **Linear decision boundary:** Logistic Regression assumes that the boundary between high- and low-performing regions can be represented approximately linearly.
* **Poor representation of nonlinear objectives:** Complex relationships between input variables may not be captured accurately.
* **Multimodal functions:** The approach can struggle when multiple isolated high-value regions exist.
* **Loss of continuous information:** Converting the output into two classes discards the magnitude of the original function values.
* **Median-dependent threshold:** The definition of "high performing" depends on the current observations and may change as additional samples are collected.
* **Random candidate sampling:** Searching 50,000 randomly generated points provides an approximation to the continuous optimization problem rather than directly optimizing the acquisition function.

---

## Overall Workflow

The complete process can be summarized as:

```text
Excel Workbook
      │
      ├── Function F1
      ├── Function F2
      ├── Function F3
      └── ...
           │
           ▼
      Extract X and y
           │
           ▼
      Calculate median(y)
           │
           ▼
      Convert y → Binary Classes
      ┌─────────────────────────┐
      │ y ≥ median → Class 1   │
      │ y < median → Class 0   │
      └─────────────────────────┘
           │
           ▼
      Train Logistic Regression
           │
           ▼
      Generate 50,000 Candidates
           │
           ▼
      Calculate P(Class = 1)
           │
           ▼
      Select Candidate with
      Highest Probability
           │
           ▼
      Set next_query
           │
           ▼
      Compile Results
           │
           ▼
      Summary DataFrame
```

### Summary

The method replaces conventional continuous surrogate regression with a **classification-based surrogate**. Rather than predicting the exact function value, Logistic Regression estimates whether a candidate point is likely to belong to a **high-performing region**.

The optimization therefore follows the principle:

> **Find the region most likely to produce an above-median function value, then evaluate the most promising candidate within that region.**

This makes the approach simple and computationally efficient, but its effectiveness depends strongly on whether the underlying black-box function can be reasonably represented by a **linear separation between high- and low-performing regions**.


In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

np.random.seed(42)

results = []

for sheet in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet)
    X = df.drop(columns=['Output']).values
    y = df['Output'].values
    n_dim = X.shape[1]
    
    # Define binary target for Logistic Regression (top quantile/median)
    threshold = np.median(y)
    y_binary = (y >= threshold).astype(int)
    
    # Train Logistic Regression model
    model = LogisticRegression(solver='lbfgs')
    model.fit(X, y_binary)
    
    # Sample candidate points across input domain [0, 1]^d
    candidates = np.random.uniform(0, 1, size=(50000, n_dim))
    
    # Predict probability of belonging to high-output class
    probs = model.predict_proba(candidates)[:, 1]
    
    # Next query point is candidate with maximum probability
    best_idx = np.argmax(probs)
    next_query = candidates[best_idx]
    max_prob = probs[best_idx]
    
    # Format next query input values
    query_str = ", ".join([f"{v:.6f}" for v in next_query])
    
    results.append({
        'Function': sheet,
        'Dimensions': n_dim,
        'Current Max Output': np.max(y),
        'Next Query Input Point': query_str
    })

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

Function  Dimensions  Current Max Output                                                         Next Query Input Point
      F1           2        7.710875e-16                                                             0.996744, 0.999561
      F2           2        6.112050e-01                                                             0.999172, 0.995664
      F3           3       -3.483500e-02                                                   0.977172, 0.978800, 0.009273
      F4           4       -4.025542e+00                                         0.020423, 0.030989, 0.041999, 0.086700
      F5           4        1.088860e+03                                         0.121200, 0.096757, 0.964209, 0.990476
      F6           5       -7.142650e-01                               0.027116, 0.620919, 0.856829, 0.999195, 0.013157
      F7           6        1.364968e+00                     0.011664, 0.151144, 0.904624, 0.163083, 0.010445, 0.282296
      F8           8        9.598482e+00

<div style="background-color: #d4edda; color: #155724; padding: 15px; border-radius: 5px; border: 1px solid #c3e6cb;">
  <strong>Success!</strong> Logistic Regression Done!
</div>

# 🎯 # Random Forest-Based Sequential Model-Based Optimization

The black-box optimization pipeline uses a **Random Forest-based Sequential Model-Based Optimization (SMBO)** framework, commonly referred to as **forest-based Bayesian optimization**.

Because the underlying function evaluations may be expensive or unknown, the Random Forest surrogate approximates the objective function and guides the search toward promising regions while balancing **exploitation** and **exploration**.

## 1. Data Extraction and Feature Mapping

### Sheet-by-Sheet Parsing

The dataset containing functions **F1 through F8** is loaded into separate data structures. Each worksheet represents an independent black-box function.

### Dimensionality Identification

The functions have different numbers of input variables:

* **F1–F2:** 2 dimensions
* **F3:** 3 dimensions
* **F4–F5:** 4 dimensions
* **F6:** 5 dimensions
* **F7:** 6 dimensions
* **F8:** 8 dimensions

Thus, each function has its own dimensionality $d$.

### Variable Matrixing

The input features ($X$) and target responses ($y$) are separated to form the training dataset:

$$
\mathcal{D}_i =
\left{
(x_1,y_1),
(x_2,y_2),
\dots,
(x_N,y_N)
\right}
$$

where:

* $x_j$ is an input vector.
* $y_j$ is the corresponding observed function output.
* $N$ is the number of available observations.

---

## 2. Surrogate Model Fitting — Random Forest Regressor

### Ensemble Learning

A Random Forest consisting of $M$ decision trees is trained on each function's dataset $\mathcal{D}_i$.

A typical configuration uses approximately:

$$
M = 100\text{--}300
$$

trees.

Each tree produces an independent prediction for a candidate point $x$.

### Mean Prediction

The Random Forest's central prediction is calculated as the average prediction across all trees:

$$
\mu(x)
======

\frac{1}{M}
\sum_{m=1}^{M}
\hat{y}_m(x)
$$

where $\hat{y}_m(x)$ represents the prediction from tree $m$.

### Uncertainty Quantification

Unlike a conventional deterministic regression model, the ensemble can provide an approximate measure of uncertainty by examining the variation between individual tree predictions:

$$
\sigma(x)
=========

\sqrt{
\frac{1}{M-1}
\sum_{m=1}^{M}
\left(
\hat{y}_m(x)-\mu(x)
\right)^2
}
$$

A large $\sigma(x)$ indicates substantial disagreement between the trees, which can be used as an indication that the region is less well understood by the surrogate.

> **Note:** This tree-to-tree standard deviation is an ensemble uncertainty heuristic rather than a calibrated posterior standard deviation in the Gaussian Process sense.

---

## 3. Candidate Domain Sampling

The optimization searches the bounded domain:

$$
[0,1]^d
$$

A large set of candidate points is generated using uniform sampling.

Typically:

$$
N_{\text{sample}} \geq 100{,}000
$$

candidate feature vectors are generated across the $d$-dimensional unit hypercube.

### Batch Prediction

All candidate points are passed through the fitted Random Forest.

For each candidate $x$, the model produces:

* Mean prediction $\mu(x)$
* Uncertainty estimate $\sigma(x)$

The candidate set therefore provides an approximation of the surrogate landscape over the entire search domain.

---

## 4. Acquisition Function Maximization — Upper Confidence Bound (UCB)

The **Upper Confidence Bound (UCB)** acquisition function combines the predicted mean with the model's uncertainty:

$$
\operatorname{UCB}(x)
=====================

\mu(x)
+
\beta\sigma(x)
$$

where $\beta$ controls the balance between exploitation and exploration.

### Exploitation

The mean prediction:

$$
\mu(x)
$$

favours regions where the surrogate predicts a high function value.

These are areas where the model believes a strong result is likely.

### Exploration

The uncertainty term:

$$
\beta\sigma(x)
$$

favours regions where the Random Forest's individual trees disagree strongly.

Such disagreement can indicate areas where the model has less confidence and where additional observations could provide useful information.

### Role of $\beta$

The parameter $\beta$ controls the exploration–exploitation balance:

* **Small $\beta$** → greater emphasis on predicted function values.
* **Large $\beta$** → greater emphasis on uncertain regions.

Thus, UCB provides a mechanism for considering both predicted performance and model uncertainty when selecting the next query.

---

## 5. Query Recommendation

After calculating the UCB value for every candidate, the next query point is selected using an argmax operation:

$$
x^*
===

\arg\max_{x\in[0,1]^d}
\operatorname{UCB}(x)
$$

The resulting $x^*$ represents the candidate with the highest acquisition score among the sampled points.

This point is exported as the recommended input configuration for the next evaluation of the actual black-box function.

---

## 6. Sequential Optimization Loop

Once the actual function is evaluated at $x^*$, the newly observed result $y^*$ is added to the existing dataset:

$$
\mathcal{D}_i
\leftarrow
\mathcal{D}_i
\cup
{(x^*,y^*)}
$$

The Random Forest is then retrained using the expanded dataset, and the acquisition process is repeated.

The complete sequential loop is:

```text
Initial Observations
        │
        ▼
Separate X and y
        │
        ▼
Train Random Forest
        │
        ├───────────────┐
        ▼               │
Generate Candidates     │
        │               │
        ▼               │
Predict μ(x) and σ(x)   │
        │               │
        ▼               │
Calculate UCB(x)        │
        │               │
        ▼               │
Select x* = argmax UCB  │
        │               │
        ▼               │
Evaluate Black Box     │
        │               │
        ▼               │
Obtain y*               │
        │               │
        └───────────────┘
          Add (x*, y*)
          and retrain
```

## Summary

The Random Forest-based SMBO approach uses **ensemble predictions as a surrogate for the unknown objective function**. The mean prediction estimates where high values are expected, while disagreement between trees provides an approximate measure of uncertainty.

The optimization strategy can therefore be summarized as:

> **Train a Random Forest surrogate, estimate both predicted performance and model uncertainty across a large candidate set, use UCB to balance exploitation and exploration, and evaluate the candidate with the highest acquisition value.**

This framework is particularly useful when the objective function is expensive to evaluate and the available dataset is too small to justify evaluating large numbers of real-world candidate points directly.


In [4]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

np.random.seed(42)

results = {}

for sheet in xls.sheet_names:
    df = pd.read_excel(excel_path, sheet_name=sheet)
    X = df.drop(columns=['Output'])
    y = df['Output']
    
    # Fit Random Forest Regressor
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X, y)
    
    # Generate dense random candidates across [0, 1] domain for inputs
    n_samples = 100000
    n_features = X.shape[1]
    candidates = np.random.uniform(0, 1, size=(n_samples, n_features))
    candidates_df = pd.DataFrame(candidates, columns=X.columns)
    
    # Upper Confidence Bound (UCB) / Expected Improvement surrogate sampling using forest variance
    preds = np.array([tree.predict(candidates) for tree in rf.estimators_])
    mean_pred = np.mean(preds, axis=0)
    std_pred = np.std(preds, axis=0)
    
    # Acquisition function: Mean + kappa * Std (balancing exploration and exploitation)
    kappa = 1.96
    acquisition = mean_pred + kappa * std_pred
    
    best_idx = np.argmax(acquisition)
    best_query = candidates_df.iloc[best_idx].to_dict()
    predicted_val = mean_pred[best_idx]
    
    # Also find current max in dataset
    current_max_row = df.loc[df['Output'].idxmax()]
    
    results[sheet] = {
        "n_inputs": n_features,
        "current_max_output": df['Output'].max(),
        "next_query": best_query,
        "predicted_output": predicted_val
    }

for f, res in results.items():
    print(f"--- {f} ---")
    print(f"Current Max Output: {res['current_max_output']:.6e}")
    print(f"Predicted Output: {res['predicted_output']:.6e}")
    print("Recommended Next Query:")
    for k, v in res['next_query'].items():
        print(f"  {k}: {v:.6f}")
    print()

--- F1 ---
Current Max Output: 7.710875e-16
Predicted Output: -1.009698e-03
Recommended Next Query:
  Input_1: 0.633530
  Input_2: 0.535775

--- F2 ---
Current Max Output: 6.112050e-01
Predicted Output: 4.462826e-01
Recommended Next Query:
  Input_1: 0.623066
  Input_2: 0.976001

--- F3 ---
Current Max Output: -3.483500e-02
Predicted Output: -1.590931e-01
Recommended Next Query:
  Input_1: 0.253420
  Input_2: 0.613758
  Input_3: 0.904660

--- F4 ---
Current Max Output: -4.025542e+00
Predicted Output: -1.618490e+01
Recommended Next Query:
  Input_1: 0.930576
  Input_2: 0.398254
  Input_3: 0.415640
  Input_4: 0.061126

--- F5 ---
Current Max Output: 1.088860e+03
Predicted Output: 6.871700e+02
Recommended Next Query:
  Input_1: 0.695668
  Input_2: 0.991208
  Input_3: 0.926107
  Input_4: 0.245121

--- F6 ---
Current Max Output: -7.142650e-01
Predicted Output: -1.008313e+00
Recommended Next Query:
  Input_1: 0.488505
  Input_2: 0.221011
  Input_3: 0.031773
  Input_4: 0.746228
  Input_5: 0.0

In [5]:
# Let's inspect each dataset to see if pure exploitation or UCB is better suited given small sample sizes.
for sheet in xls.sheet_names:
    df = pd.read_excel(excel_path, sheet_name=sheet)
    best_row = df.loc[df['Output'].idxmax()]
    print(f"=== {sheet} ===")
    print(f"Max row:\n{best_row}")

=== F1 ===
Max row:
Input_1    7.310240e-01
Input_2    7.330000e-01
Output     7.710875e-16
Name: 2, dtype: float64
=== F2 ===
Max row:
Input_1    0.702637
Input_2    0.926564
Output     0.611205
Name: 9, dtype: float64
=== F3 ===
Max row:
Input_1    0.492581
Input_2    0.611593
Input_3    0.340176
Output    -0.034835
Name: 3, dtype: float64
=== F4 ===
Max row:
Input_1    0.577766
Input_2    0.428772
Input_3    0.425826
Input_4    0.249007
Output    -4.025542
Name: 27, dtype: float64
=== F5 ===
Max row:
Input_1       0.224189
Input_2       0.846480
Input_3       0.879484
Input_4       0.878516
Output     1088.859618
Name: 15, dtype: float64
=== F6 ===
Max row:
Input_1    0.728186
Input_2    0.154693
Input_3    0.732552
Input_4    0.693997
Input_5    0.056401
Output    -0.714265
Name: 0, dtype: float64
=== F7 ===
Max row:
Input_1    0.057896
Input_2    0.491672
Input_3    0.247422
Input_4    0.218118
Input_5    0.420428
Input_6    0.730970
Output     1.364968
Name: 6, dtype: float64
===

In [6]:
# Let's perform standard Bayesian Optimization using Random Forest (Forest-based Bayesian Optimization / Sequential Model-Based Optimization - SMBO) with Upper Confidence Bound (UCB).

final_recommendations = {}

for sheet in xls.sheet_names:
    df = pd.read_excel(excel_path, sheet_name=sheet)
    X = df.drop(columns=['Output'])
    y = df['Output']
    
    # Train Random Forest
    rf = RandomForestRegressor(n_estimators=300, max_features='sqrt', random_state=42)
    rf.fit(X, y)
    
    # Sample 200,000 points uniformly from domain [0, 1]^d
    n_samples = 200000
    n_dim = X.shape[1]
    candidates = np.random.uniform(0, 1, size=(n_samples, n_dim))
    
    # Get tree predictions
    tree_preds = np.array([tree.predict(candidates) for tree in rf.estimators_])
    mean_pred = np.mean(tree_preds, axis=0)
    std_pred = np.std(tree_preds, axis=0)
    
    # Upper Confidence Bound (UCB): Mean + beta * Std
    # Setting beta = 1.5 balances exploiting known high regions & exploring uncertain regions
    beta = 1.5
    ucb_scores = mean_pred + beta * std_pred
    
    best_idx = np.argmax(ucb_scores)
    query = candidates[best_idx]
    
    final_recommendations[sheet] = {
        "inputs": X.columns.tolist(),
        "query": query,
        "current_max": y.max(),
        "pred_mean": mean_pred[best_idx],
        "pred_ucb": ucb_scores[best_idx]
    }

for f, res in final_recommendations.items():
    print(f"{f}:")
    for col, val in zip(res["inputs"], res["query"]):
        print(f"  {col}: {val:.6f}")

F1:
  Input_1: 0.311635
  Input_2: 0.636866
F2:
  Input_1: 0.631672
  Input_2: 0.946033
F3:
  Input_1: 0.143948
  Input_2: 0.539062
  Input_3: 0.373001
F4:
  Input_1: 0.499797
  Input_2: 0.427206
  Input_3: 0.432212
  Input_4: 0.240915
F5:
  Input_1: 0.200312
  Input_2: 0.832589
  Input_3: 0.861807
  Input_4: 0.866736
F6:
  Input_1: 0.084334
  Input_2: 0.170551
  Input_3: 0.690843
  Input_4: 0.708093
  Input_5: 0.036388
F7:
  Input_1: 0.011676
  Input_2: 0.706700
  Input_3: 0.192961
  Input_4: 0.083524
  Input_5: 0.442081
  Input_6: 0.917694
F8:
  Input_1: 0.121637
  Input_2: 0.149629
  Input_3: 0.023516
  Input_4: 0.058765
  Input_5: 0.989582
  Input_6: 0.765757
  Input_7: 0.148112
  Input_8: 0.857093


In [7]:
for f, res in final_recommendations.items():
    print(f"**{f} Recommended Query**")
    for col, val in zip(res["inputs"], res["query"]):
        print(f"  * {col}: {val:.6f}")
    print(f"  * Current Best Output: {res['current_max']:.6e}")
    print()

**F1 Recommended Query**
  * Input_1: 0.311635
  * Input_2: 0.636866
  * Current Best Output: 7.710875e-16

**F2 Recommended Query**
  * Input_1: 0.631672
  * Input_2: 0.946033
  * Current Best Output: 6.112050e-01

**F3 Recommended Query**
  * Input_1: 0.143948
  * Input_2: 0.539062
  * Input_3: 0.373001
  * Current Best Output: -3.483500e-02

**F4 Recommended Query**
  * Input_1: 0.499797
  * Input_2: 0.427206
  * Input_3: 0.432212
  * Input_4: 0.240915
  * Current Best Output: -4.025542e+00

**F5 Recommended Query**
  * Input_1: 0.200312
  * Input_2: 0.832589
  * Input_3: 0.861807
  * Input_4: 0.866736
  * Current Best Output: 1.088860e+03

**F6 Recommended Query**
  * Input_1: 0.084334
  * Input_2: 0.170551
  * Input_3: 0.690843
  * Input_4: 0.708093
  * Input_5: 0.036388
  * Current Best Output: -7.142650e-01

**F7 Recommended Query**
  * Input_1: 0.011676
  * Input_2: 0.706700
  * Input_3: 0.192961
  * Input_4: 0.083524
  * Input_5: 0.442081
  * Input_6: 0.917694
  * Current Best

<div style="background-color: #d4edda; color: #155724; padding: 15px; border-radius: 5px; border: 1px solid #c3e6cb;">
  <strong>Success!</strong> Random Forest-based Sequential Model-Based Optimization (SMBO) framework Done!
</div>

# 🎯 Surrogate-Based Optimization Using Support Vector Regression

This script implements a **Surrogate-Based Optimization pipeline** using a **Support Vector Machine (SVM)** to approximate black-box functions and search for input coordinates that maximize their outputs.

The approach uses **Support Vector Regression (SVR)** with a **Radial Basis Function (RBF)** kernel as the surrogate model, followed by a multi-start numerical optimization procedure to locate the predicted maximum.

## 1. Data Loading and Dynamic Column Parsing

The script loads an Excel workbook and processes each worksheet independently.

### Excel Workbook Processing

```python
pd.ExcelFile(excel_path)
```

opens the workbook and allows the script to iterate through every worksheet:

```python
sheet_names
```

Each sheet is treated as a separate **black-box function dataset**.

### Dynamic Feature Detection

The input columns are detected dynamically:

```python
feature_cols = [
    col for col in df.columns
    if col.startswith('Input')
]
```

This automatically identifies columns such as:

```text
Input_1
Input_2
Input_3
...
```

regardless of the dimensionality of the function.

The resulting datasets are separated into:

* `X` — input coordinates/features
* `y` — observed target/output values

---

## 2. Feature Normalization and Surrogate Model Training

### Feature Scaling

The script uses `StandardScaler()` to normalize the input features.

The transformation standardizes each feature to approximately:

```text
mean = 0
standard deviation = 1
```

This is particularly important for SVM-based models because their kernel calculations depend on distances between observations.

Without normalization, a feature with a substantially larger numerical scale could have a disproportionate influence on the model.

### Support Vector Regression

The surrogate model is configured as:

```python
SVR(
    kernel='rbf',
    C=10.0,
    epsilon=0.01
)
```

The SVR approximates the unknown black-box function using the available observations.

The **Radial Basis Function (RBF)** kernel allows the surrogate to model nonlinear relationships between the input coordinates and the output.

This makes the model considerably more flexible than a simple linear regression model.

### Surrogate Role

Once trained, the SVR provides an inexpensive approximation of the underlying black-box function:

```text
Input coordinates
       │
       ▼
   Trained SVR
       │
       ▼
Predicted function value
```

The optimizer can therefore search the surrogate model rather than repeatedly evaluating the expensive original function.

---

## 3. Global Search via Multi-Start Gradient Optimization

Because black-box functions may contain complex nonlinear surfaces and multiple local peaks, a single optimization run can converge to a suboptimal local solution.

The script addresses this using a **multi-start optimization strategy**.

### Multi-Start Strategy

The optimizer performs:

```python
range(50)
```

independent optimization runs.

For each run, a random starting point is generated uniformly within the unit hypercube:

```text
x₀ ∈ [0, 1]^d
```

where `d` is the number of input dimensions.

Using multiple starting locations increases the likelihood of discovering different regions of the surrogate landscape.

---

### Objective Function

The trained SVR is wrapped inside an objective function:

```python
def objective(x):
    return -svr.predict(...)
```

The negative sign is important because `scipy.optimize.minimize()` performs **minimization**, whereas the goal is to **maximize** the predicted function value.

Therefore:

```text
Maximize SVR(x)
        ↓
Minimize -SVR(x)
```

The point producing the smallest negative prediction corresponds to the largest positive SVR prediction.

---

### L-BFGS-B Optimization

Each starting point is optimized using:

```python
minimize(
    ...,
    method='L-BFGS-B',
    bounds=[(0, 1)] * n_dim
)
```

**L-BFGS-B** is a bounded quasi-Newton optimization algorithm.

The bounds ensure that every candidate remains within the original search domain:

```text
0 ≤ xᵢ ≤ 1
```

for every dimension `i`.

The optimization process therefore searches for a local maximum of the SVR surrogate while respecting the original function's domain.

---

### Best-Value Tracking

After each optimization run, the resulting prediction is compared against the best solution found so far.

Conceptually:

```text
For each of 50 starting points:

    Optimize -SVR(x)

    ↓

    Obtain local optimum

    ↓

    Compare predicted value with best_pred

    ↓

    Keep the best solution
```

The script ultimately stores:

* `best_pred` — highest predicted output found by the surrogate
* `best_x` — input coordinates corresponding to that prediction

This `best_x` becomes the recommended **next query point** for evaluating the actual black-box function.

---

## 4. Result Aggregation and Output Formatting

After optimization, the script formats the optimal coordinates:

```text
best_x
```

with values rounded to **4 decimal places**.

For each worksheet, the script creates a summary containing:

| Field                    | Description                                              |
| ------------------------ | -------------------------------------------------------- |
| **Function**             | Identifier for the black-box function                    |
| **Dimensions**           | Number of input dimensions                               |
| **Samples**              | Number of observations used to train the surrogate       |
| **Max Observed Output**  | Highest output actually observed in the dataset          |
| **Predicted Max Output** | Highest output predicted by the SVR surrogate            |
| **Next Query**           | Input coordinates corresponding to the predicted maximum |

The results from all worksheets are then combined into a pandas DataFrame and displayed as a clean summary table.

---

# Overall Workflow

The complete optimization pipeline can be summarized as:

```text
Excel Workbook
      │
      ├── Function F1
      ├── Function F2
      ├── Function F3
      └── ...
           │
           ▼
     Extract X and y
           │
           ▼
     Detect Input Columns
           │
           ▼
      StandardScaler
           │
           ▼
     Train RBF-SVR
           │
           ▼
   Generate Random Start
       Points × 50
           │
           ▼
      L-BFGS-B
     Optimization
           │
           ▼
     Minimize -SVR(x)
           │
           ▼
   Track Best Predicted
        Solution
           │
           ▼
      best_x / best_pred
           │
           ▼
   Format & Aggregate
        Results
           │
           ▼
      Summary DataFrame
```

## Summary

The method uses **Support Vector Regression with an RBF kernel** as a nonlinear surrogate for each black-box function. The trained surrogate is then searched using **50 independently initialized L-BFGS-B optimizations** within the bounded domain `[0, 1]^d`.

The overall strategy is:

> **Learn an approximate nonlinear representation of the black-box function, search that surrogate for its predicted maximum, and use the corresponding coordinates as the next point to evaluate on the actual function.**

Compared with a linear classification-based surrogate, the RBF-SVR approach can represent substantially more complex nonlinear relationships. However, the predicted maximum remains dependent on the quality of the surrogate and on whether the multi-start optimization successfully explores the relevant regions of the surrogate landscape.


In [8]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize

xls = pd.ExcelFile(excel_path)

results = []

for sheet in xls.sheet_names:
    df = pd.read_excel(excel_path, sheet_name=sheet)
    feature_cols = [col for col in df.columns if col.startswith('Input')]
    X = df[feature_cols].values
    y = df['Output'].values
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Train Support Vector Regressor (RBF Kernel)
    svr = SVR(kernel='rbf', C=10.0, epsilon=0.01)
    svr.fit(X_scaled, y)
    
    # Multi-start optimization on the surrogate SVR model to find optimal query
    best_pred = -np.inf
    best_x = None
    
    # Number of random restarts to avoid local maxima
    np.random.seed(42)
    n_dim = X.shape[1]
    
    for _ in range(50):
        x0 = np.random.uniform(0, 1, size=n_dim)
        
        def objective(x):
            x_scaled = scaler.transform(x.reshape(1, -1))
            return -svr.predict(x_scaled)[0]
            
        bounds = [(0, 1) for _ in range(n_dim)]
        res = minimize(objective, x0, bounds=bounds, method='L-BFGS-B')
        
        if -res.fun > best_pred:
            best_pred = -res.fun
            best_x = res.x

    # Format output query point
    query_str = ", ".join([f"{val:.4f}" for val in best_x])
    
    results.append({
        'Function': sheet,
        'Inputs': len(feature_cols),
        'Data Points': len(df),
        'Max Observed Output': np.max(y),
        'Predicted Max Output': best_pred,
        'Recommended Next Query Vector [0, 1]': f"[{query_str}]"
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

Function  Inputs  Data Points  Max Observed Output  Predicted Max Output                             Recommended Next Query Vector [0, 1]
      F1       2           10         7.710875e-16             -0.001803                                                 [0.3745, 0.9507]
      F2       2           10         6.112050e-01              0.913694                                                 [0.8104, 1.0000]
      F3       3           15        -3.483500e-02             -0.018368                                         [0.3023, 0.1873, 0.5053]
      F4       4           30        -4.025542e+00             -1.007575                                 [0.4242, 0.4299, 0.4130, 0.4032]
      F5       4           20         1.088860e+03             86.493844                                 [0.2018, 0.8687, 0.7747, 0.9645]
      F6       5           20        -7.142650e-01             -0.489174                         [0.5090, 0.3832, 0.5222, 0.7044, 0.1909]
      F7       6           30     

In [9]:
# Print precise values for optimal queries
for res in results:
    print(f"--- {res['Function']} ---")
    print(f"Max Observed: {res['Max Observed Output']}")
    print(f"Predicted Max: {res['Predicted Max Output']}")
    print(f"Query Vector: {res['Recommended Next Query Vector [0, 1]']}\n")

--- F1 ---
Max Observed: 7.710875e-16
Predicted Max: -0.0018030314999996138
Query Vector: [0.3745, 0.9507]

--- F2 ---
Max Observed: 0.611205
Predicted Max: 0.9136938008939367
Query Vector: [0.8104, 1.0000]

--- F3 ---
Max Observed: -0.034835
Predicted Max: -0.018368168466710244
Query Vector: [0.3023, 0.1873, 0.5053]

--- F4 ---
Max Observed: -4.025542
Predicted Max: -1.0075747774985082
Query Vector: [0.4242, 0.4299, 0.4130, 0.4032]

--- F5 ---
Max Observed: 1088.859618
Predicted Max: 86.49384398910658
Query Vector: [0.2018, 0.8687, 0.7747, 0.9645]

--- F6 ---
Max Observed: -0.714265
Predicted Max: -0.48917356246352917
Query Vector: [0.5090, 0.3832, 0.5222, 0.7044, 0.1909]

--- F7 ---
Max Observed: 1.364968
Predicted Max: 1.5829538397845622
Query Vector: [0.0000, 0.3730, 0.2890, 0.0711, 0.3553, 0.7947]

--- F8 ---
Max Observed: 9.598482
Predicted Max: 10.409850712266149
Query Vector: [0.1866, 0.2756, 0.1956, 0.3153, 0.5139, 0.4871, 0.4034, 0.5587]



### Why SVR Underperforms for Black-Box Optimization

Although Support Vector Regression (SVR) can provide a useful nonlinear surrogate model, several characteristics make it less effective for **black-box optimization**, particularly when evaluations are expensive and the objective contains sharp or isolated peaks.

#### 1. No Uncertainty Estimates

SVR produces **point predictions** rather than an explicit probability distribution or confidence interval.

As a result, the model cannot directly quantify how uncertain it is in different regions of the search space. The optimizer therefore has little information about whether an apparently poor region is:

* genuinely low-performing, or
* simply insufficiently explored.

This tends to push the optimization strategy toward **exploitation**—searching areas that the surrogate already predicts to be promising—while providing no principled mechanism for **exploration** of poorly sampled regions.

By contrast, probabilistic surrogate models such as Gaussian Processes provide both a predicted mean and an uncertainty estimate, which can be incorporated into an acquisition function.

---

#### 2. Sensitivity to Hyperparameters

SVR performance depends strongly on the choice of hyperparameters, including:

* **$C$** — controls the penalty for prediction errors.
* **$\gamma$** — controls the influence or effective width of the RBF kernel.
* **$\epsilon$** — defines the tolerance region around the regression function.

Poorly selected hyperparameters can produce either:

* **Overfitting**, where the surrogate follows the observed data too closely and creates misleading local features.
* **Over-smoothing**, where important variations and peaks in the objective landscape are lost.

Either situation can cause the subsequent optimizer to converge toward an unproductive region or become trapped around a local optimum.

---

#### 3. Smoothing of Extreme Peaks

SVR can struggle with **isolated extreme observations**, particularly when the majority of the dataset contains substantially smaller values.

This is illustrated by **F5**, where the observed function contains an extreme peak of approximately:

$$
1088
$$

The SVR surrogate may smooth this isolated high-value region rather than reproducing its magnitude accurately. Consequently, the predicted maximum can be substantially lower than the true observed peak.

This is particularly problematic for optimization because the objective is not simply to model the average behavior of the function—it is to **find its extreme values**.

---

#### Overall Issue

The fundamental limitation is that standard SVR is primarily designed as a **predictive regression model**, not as a sequential decision-making framework.

Its workflow is essentially:

```text
Observed Data
     │
     ▼
Train SVR
     │
     ▼
Predict Function Values
     │
     ▼
Optimize Predicted Surface
```

There is no inherent mechanism for asking:

> **"Where should I sample next because the model is uncertain and the region could contain a better solution?"**

This is a key distinction from **Bayesian Optimization with Gaussian Processes**, where model uncertainty can be explicitly incorporated into acquisition functions such as **Expected Improvement (EI)**, **Upper Confidence Bound (UCB)**, or **Probability of Improvement (PI)**.

Therefore, for small-sample black-box problems with expensive evaluations, sharp peaks, or substantial unexplored regions, **Gaussian Process-based Bayesian Optimization is generally better suited to sequential optimization than standard SVR**.


<div style="background-color: #d4edda; color: #155724; padding: 15px; border-radius: 5px; border: 1px solid #c3e6cb;">
  <strong>Success!</strong> Support Vector Machine (SVM) Done!
</div>

# 🧠 Neural Networks with MLPRegressor

1. **Surrogate Learning:** Fits a **Multi-Layer Perceptron (MLP)** neural network with two hidden layers of 64 nodes each to approximate the underlying black-box function:

   $$
   f(\mathbf{x})
   $$

   The neural network learns a nonlinear mapping between the input coordinates $\mathbf{x}$ and the observed function output.

2. **`solver='lbfgs'`:** Uses a **quasi-Newton optimization algorithm** to train the neural network. `L-BFGS` is particularly well suited to small datasets and can converge faster and more reliably than stochastic gradient-based solvers when the number of observations is small (approximately $N \leq 40$).

   For the small datasets considered here, this makes `lbfgs` a practical choice for fitting the neural-network surrogate.


In [10]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from scipy.optimize import minimize

# Set random seed for reproducibility
np.random.seed(42)

xls = pd.ExcelFile(excel_path)

results = {}

for sheet in xls.sheet_names:
    df = pd.read_excel(excel_path, sheet_name=sheet)
    
    # Identify input features and output
    X = df.filter(like='Input').values
    y = df['Output'].values
    n_inputs = X.shape[1]
    
    # Train Neural Network (MLPRegressor) surrogate model
    # Tuning hyperparameters slightly depending on small dataset sizes
    model = MLPRegressor(
        hidden_layer_sizes=(64, 64),
        activation='relu',
        solver='lbfgs',      # lbfgs works very well on small datasets
        max_iter=2000,
        random_state=42
    )
    model.fit(X, y)
    
    # We want to MAXIMIZE output y, so we MINIMIZE -model.predict(x)
    def objective_func(x):
        return -model.predict(x.reshape(1, -1))[0]
    
    # Run multiple random restarts within the domain bounds [0, 1] (or inferred domain)
    # Checking min and max of inputs to set appropriate bounds
    bounds = [(0.0, 1.0)] * n_inputs
    
    best_val = float('inf')
    best_x = None
    
    # Perform 50 random restarts
    for _ in range(50):
        x0 = np.random.uniform(0.0, 1.0, size=n_inputs)
        res = minimize(objective_func, x0, method='L-BFGS-B', bounds=bounds)
        if res.fun < best_val:
            best_val = res.fun
            best_x = res.x
            
    results[sheet] = {
        'n_inputs': n_inputs,
        'best_inputs': np.round(best_x, 6),
        'predicted_max_output': np.round(-best_val, 6),
        'current_max_in_data': np.round(y.max(), 6)
    }

for fn, res in results.items():
    print(f"=== {fn} ===")
    print(f"Inputs count: {res['n_inputs']}")
    print(f"Current Max Output in Data: {res['current_max_in_data']}")
    print(f"NN Predicted Max Output: {res['predicted_max_output']}")
    print(f"Optimal Inputs: {list(res['best_inputs'])}\n")

=== F1 ===
Inputs count: 2
Current Max Output in Data: 0.0
NN Predicted Max Output: 0.007655
Optimal Inputs: [np.float64(0.725287), np.float64(0.573756)]

=== F2 ===
Inputs count: 2
Current Max Output in Data: 0.611205
NN Predicted Max Output: 2.377145
Optimal Inputs: [np.float64(0.0), np.float64(1.0)]

=== F3 ===
Inputs count: 3
Current Max Output in Data: -0.034835
NN Predicted Max Output: 0.135583
Optimal Inputs: [np.float64(0.227981), np.float64(0.0), np.float64(0.836454)]

=== F4 ===
Inputs count: 4
Current Max Output in Data: -4.025542
NN Predicted Max Output: -0.563708
Optimal Inputs: [np.float64(0.43701), np.float64(0.41688), np.float64(0.312658), np.float64(0.389712)]

=== F5 ===
Inputs count: 4
Current Max Output in Data: 1088.859618
NN Predicted Max Output: 3995.701421
Optimal Inputs: [np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(1.0)]

=== F6 ===
Inputs count: 5
Current Max Output in Data: -0.714265
NN Predicted Max Output: -0.301544
Optimal Inputs: [np.flo

<div style="background-color: #d4edda; color: #155724; padding: 15px; border-radius: 5px; border: 1px solid #c3e6cb;">
  <strong>Success!</strong> MLPRegressor Done!
</div>